# Lab Exercise: SQL Analysis with Polars

In this lab, you'll practice SQL queries using Polars' built-in SQL functionality. Complete each exercise by writing the appropriate SQL query.

In [7]:
# Setup - Run this cell first
import polars as pl

# Load data
airlines = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_airlines.csv')
airports = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_airports.csv', null_values="NA")
flights = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_flights.csv', null_values="NA")
planes = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_planes.csv', null_values="NA")
weather = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_weather.csv', null_values="NA", infer_schema_length=1000)
flights = flights.with_columns(pl.col("time_hour").str.strptime(pl.Datetime))
weather = weather.with_columns(pl.col("time_hour").str.strptime(pl.Datetime))

# Create SQL context
ctx = pl.SQLContext(
    airlines=airlines,
    airports=airports,
    flights=flights,
    planes=planes,
    weather=weather,
    eager_execution=True
)

print("Setup complete! Tables available:")
print(ctx.execute("SHOW TABLES"))

Setup complete! Tables available:
shape: (5, 1)
┌──────────┐
│ name     │
│ ---      │
│ str      │
╞══════════╡
│ airlines │
│ airports │
│ flights  │
│ planes   │
│ weather  │
└──────────┘


/tmp/ipython-input-3794928486.py:14: DeprecationWarning: The argument `eager_execution` for `SQLContext.__init__` is deprecated. It has been renamed to `eager`.
  ctx = pl.SQLContext(


## Exercise 1: Basic Queries

### 1.1 Find all unique carriers in the airlines table

In [8]:
# Write your SQL query here
result = ctx.execute("""
    SELECT *
    FROM airlines
""")

print(result)

shape: (16, 2)
┌─────────┬────────────────────────┐
│ carrier ┆ name                   │
│ ---     ┆ ---                    │
│ str     ┆ str                    │
╞═════════╪════════════════════════╡
│ 9E      ┆ Endeavor Air Inc.      │
│ AA      ┆ American Airlines Inc. │
│ AS      ┆ Alaska Airlines Inc.   │
│ B6      ┆ JetBlue Airways        │
│ DL      ┆ Delta Air Lines Inc.   │
│ …       ┆ …                      │
│ UA      ┆ United Air Lines Inc.  │
│ US      ┆ US Airways Inc.        │
│ VX      ┆ Virgin America         │
│ WN      ┆ Southwest Airlines Co. │
│ YV      ┆ Mesa Airlines Inc.     │
└─────────┴────────────────────────┘


### 1.2 Find the top 10 destinations by number of flights

In [12]:
# Write your SQL query here
result = ctx.execute("""
    SELECT dest, COUNT(*) as flight_count
    FROM flights
    GROUP BY dest
    ORDER BY flight_count DESC
    LIMIT 10
""")

print(result)

shape: (10, 2)
┌──────┬──────────────┐
│ dest ┆ flight_count │
│ ---  ┆ ---          │
│ str  ┆ u32          │
╞══════╪══════════════╡
│ ORD  ┆ 17283        │
│ ATL  ┆ 17215        │
│ LAX  ┆ 16174        │
│ BOS  ┆ 15508        │
│ MCO  ┆ 14082        │
│ CLT  ┆ 14064        │
│ SFO  ┆ 13331        │
│ FLL  ┆ 12055        │
│ MIA  ┆ 11728        │
│ DCA  ┆ 9705         │
└──────┴──────────────┘


### 1.3 Find all flights that departed more than 2 hours late (120 minutes)

In [13]:
# Write your SQL query here
result = ctx.execute("""
    SELECT COUNT(*) as flight_count
    FROM flights
    WHERE dep_delay > 120
""")

print(result)

shape: (1, 1)
┌──────────────┐
│ flight_count │
│ ---          │
│ u32          │
╞══════════════╡
│ 9723         │
└──────────────┘


## Exercise 2: Aggregation

### 2.1 Calculate the average departure delay for each origin airport

In [17]:
# Write your SQL query here
result = ctx.execute("""
    SELECT origin, AVG(dep_delay) as avg_delay
    FROM flights
    WHERE dep_delay IS NOT NULL
    GROUP BY origin
    ORDER BY avg_delay DESC
""")

print(result)

shape: (3, 2)
┌────────┬───────────┐
│ origin ┆ avg_delay │
│ ---    ┆ ---       │
│ str    ┆ f64       │
╞════════╪═══════════╡
│ EWR    ┆ 15.107954 │
│ JFK    ┆ 12.112159 │
│ LGA    ┆ 10.346876 │
└────────┴───────────┘


### 2.2 Find the busiest month of the year

Count the number of flights per month and find which month has the most flights.

In [26]:
print(ctx.execute("""
    SELECT year, month, day, dep_time, arr_time
    FROM flights
    LIMIT 5
"""))

result = ctx.execute("""
WITH monthly AS (
    SELECT
        month,
        COUNT(*) AS flight_count
    FROM flights
    GROUP BY month
),
max_count AS (
    SELECT MAX(flight_count) AS max_flights FROM monthly
)
SELECT m.month, m.flight_count
FROM monthly AS m
JOIN max_count AS x
  ON m.flight_count = x.max_flights
ORDER BY m.month
""")

print("Busiest month(s):")
print(result)

shape: (5, 5)
┌──────┬───────┬─────┬──────────┬──────────┐
│ year ┆ month ┆ day ┆ dep_time ┆ arr_time │
│ ---  ┆ ---   ┆ --- ┆ ---      ┆ ---      │
│ i64  ┆ i64   ┆ i64 ┆ i64      ┆ i64      │
╞══════╪═══════╪═════╪══════════╪══════════╡
│ 2013 ┆ 1     ┆ 1   ┆ 517      ┆ 830      │
│ 2013 ┆ 1     ┆ 1   ┆ 533      ┆ 850      │
│ 2013 ┆ 1     ┆ 1   ┆ 542      ┆ 923      │
│ 2013 ┆ 1     ┆ 1   ┆ 544      ┆ 1004     │
│ 2013 ┆ 1     ┆ 1   ┆ 554      ┆ 812      │
└──────┴───────┴─────┴──────────┴──────────┘
Busiest month(s):
shape: (1, 2)
┌───────┬──────────────┐
│ month ┆ flight_count │
│ ---   ┆ ---          │
│ i64   ┆ u32          │
╞═══════╪══════════════╡
│ 7     ┆ 29425        │
└───────┴──────────────┘


### 2.3 Calculate the on-time performance rate for each carrier

Consider a flight on-time if the departure delay is <= 15 minutes.

In [28]:
result = ctx.execute(
    """
    WITH per_carrier AS (
        SELECT
            carrier,
            COUNT(*) AS total_flights,
            SUM(CASE WHEN dep_delay <= 15 THEN 1 ELSE 0 END) AS on_time_flights
        FROM flights
        WHERE dep_delay IS NOT NULL
        GROUP BY carrier
    )
    SELECT
        a.name AS airline,
        ROUND((on_time_flights * 100.0) / total_flights, 2) AS on_time_pct
    FROM per_carrier AS c
    LEFT JOIN airlines AS a
      ON a.carrier = c.carrier
    ORDER BY on_time_pct DESC, airline
    """
)

print(result)

shape: (16, 2)
┌─────────────────────────────┬─────────────┐
│ airline                     ┆ on_time_pct │
│ ---                         ┆ ---         │
│ str                         ┆ f64         │
╞═════════════════════════════╪═════════════╡
│ Hawaiian Airlines Inc.      ┆ 92.98       │
│ US Airways Inc.             ┆ 87.82       │
│ Alaska Airlines Inc.        ┆ 86.8        │
│ American Airlines Inc.      ┆ 84.07       │
│ Delta Air Lines Inc.        ┆ 83.68       │
│ …                           ┆ …           │
│ AirTran Airways Corporation ┆ 73.33       │
│ Southwest Airlines Co.      ┆ 73.1        │
│ Frontier Airlines Inc.      ┆ 71.85       │
│ Mesa Airlines Inc.          ┆ 71.38       │
│ ExpressJet Airlines Inc.    ┆ 69.54       │
└─────────────────────────────┴─────────────┘


## Exercise 3: Joins

### 3.1 List all flights with their airline names (not just carrier codes)

Show the first 20 flights with carrier code, airline name, flight number, origin, and destination.

In [29]:
result = ctx.execute(
    """
    SELECT
        f.carrier                       AS carrier,
        a.name                          AS airline,
        f.flight                        AS flight_number,
        f.origin,
        f.dest                          AS destination
    FROM flights AS f
    LEFT JOIN airlines AS a
      ON a.carrier = f.carrier
    ORDER BY f.year, f.month, f.day, f.dep_time, f.flight
    LIMIT 20
    """
)

print(result)

shape: (20, 5)
┌─────────┬────────────────────────┬───────────────┬────────┬─────────────┐
│ carrier ┆ airline                ┆ flight_number ┆ origin ┆ destination │
│ ---     ┆ ---                    ┆ ---           ┆ ---    ┆ ---         │
│ str     ┆ str                    ┆ i64           ┆ str    ┆ str         │
╞═════════╪════════════════════════╪═══════════════╪════════╪═════════════╡
│ UA      ┆ United Air Lines Inc.  ┆ 1545          ┆ EWR    ┆ IAH         │
│ UA      ┆ United Air Lines Inc.  ┆ 1714          ┆ LGA    ┆ IAH         │
│ AA      ┆ American Airlines Inc. ┆ 1141          ┆ JFK    ┆ MIA         │
│ B6      ┆ JetBlue Airways        ┆ 725           ┆ JFK    ┆ BQN         │
│ DL      ┆ Delta Air Lines Inc.   ┆ 461           ┆ LGA    ┆ ATL         │
│ …       ┆ …                      ┆ …             ┆ …      ┆ …           │
│ UA      ┆ United Air Lines Inc.  ┆ 1187          ┆ EWR    ┆ LAS         │
│ B6      ┆ JetBlue Airways        ┆ 1806          ┆ JFK    ┆ BOS        

### 3.2 Find the average age of planes for each carrier

Hint: The planes table has a `year` column for manufacture year. Calculate age based on 2013.

In [30]:
result = ctx.execute(
    """
    WITH fleet AS (
        SELECT DISTINCT
            f.carrier,
            f.tailnum,
            p.year AS manuf_year
        FROM flights AS f
        INNER JOIN planes AS p
            ON p.tailnum = f.tailnum
        WHERE p.year IS NOT NULL
    ),
    stats AS (
        SELECT
            carrier,
            AVG(2013 - manuf_year) AS avg_plane_age_years,
            COUNT(*) AS unique_planes
        FROM fleet
        GROUP BY carrier
    )
    SELECT
        s.carrier,
        a.name AS airline,
        ROUND(s.avg_plane_age_years, 2) AS avg_plane_age_years,
        s.unique_planes
    FROM stats AS s
    LEFT JOIN airlines AS a
      ON a.carrier = s.carrier
    ORDER BY avg_plane_age_years DESC, s.carrier
    """
)

print(result)

shape: (16, 4)
┌─────────┬────────────────────────┬─────────────────────┬───────────────┐
│ carrier ┆ airline                ┆ avg_plane_age_years ┆ unique_planes │
│ ---     ┆ ---                    ┆ ---                 ┆ ---           │
│ str     ┆ str                    ┆ f64                 ┆ u32           │
╞═════════╪════════════════════════╪═════════════════════╪═══════════════╡
│ MQ      ┆ Envoy Air              ┆ 35.5                ┆ 4             │
│ AA      ┆ American Airlines Inc. ┆ 25.4                ┆ 164           │
│ DL      ┆ Delta Air Lines Inc.   ┆ 17.67               ┆ 609           │
│ UA      ┆ United Air Lines Inc.  ┆ 13.05               ┆ 589           │
│ WN      ┆ Southwest Airlines Co. ┆ 11.01               ┆ 569           │
│ …       ┆ …                      ┆ …                   ┆ …             │
│ B6      ┆ JetBlue Airways        ┆ 6.5                 ┆ 187           │
│ AS      ┆ Alaska Airlines Inc.   ┆ 5.16                ┆ 83            │
│ F9      

### 3.3 Find flights that experienced both departure delays and bad weather

Join flights with weather data and find flights where departure delay > 30 minutes and either wind_speed > 20 or precip > 0.1

In [32]:
# First, explore the weather table structure
preview = ctx.execute(
    """
    SELECT *
    FROM weather
    LIMIT 5
    """
)
print(preview)

# Now write your join query
result = ctx.execute(
    """
    SELECT
        f.carrier,
        a.name AS airline,
        f.flight,
        f.origin,
        f.dest,
        f.dep_delay,
        w.wind_speed,
        w.precip,
        f.time_hour
    FROM flights AS f
    INNER JOIN weather AS w
      ON f.origin = w.origin
     AND f.time_hour = w.time_hour
    LEFT JOIN airlines AS a
      ON a.carrier = f.carrier
    WHERE f.dep_delay > 30
      AND (w.wind_speed > 20 OR w.precip > 0.1)
    ORDER BY f.dep_delay DESC, f.time_hour
    """
)

print(result)

shape: (5, 15)
┌────────┬──────┬───────┬─────┬───┬────────┬──────────┬───────┬─────────────────────────┐
│ origin ┆ year ┆ month ┆ day ┆ … ┆ precip ┆ pressure ┆ visib ┆ time_hour               │
│ ---    ┆ ---  ┆ ---   ┆ --- ┆   ┆ ---    ┆ ---      ┆ ---   ┆ ---                     │
│ str    ┆ i64  ┆ i64   ┆ i64 ┆   ┆ f64    ┆ f64      ┆ f64   ┆ datetime[μs, UTC]       │
╞════════╪══════╪═══════╪═════╪═══╪════════╪══════════╪═══════╪═════════════════════════╡
│ EWR    ┆ 2013 ┆ 1     ┆ 1   ┆ … ┆ 0.0    ┆ 1012.0   ┆ 10.0  ┆ 2013-01-01 06:00:00 UTC │
│ EWR    ┆ 2013 ┆ 1     ┆ 1   ┆ … ┆ 0.0    ┆ 1012.3   ┆ 10.0  ┆ 2013-01-01 07:00:00 UTC │
│ EWR    ┆ 2013 ┆ 1     ┆ 1   ┆ … ┆ 0.0    ┆ 1012.5   ┆ 10.0  ┆ 2013-01-01 08:00:00 UTC │
│ EWR    ┆ 2013 ┆ 1     ┆ 1   ┆ … ┆ 0.0    ┆ 1012.2   ┆ 10.0  ┆ 2013-01-01 09:00:00 UTC │
│ EWR    ┆ 2013 ┆ 1     ┆ 1   ┆ … ┆ 0.0    ┆ 1011.9   ┆ 10.0  ┆ 2013-01-01 10:00:00 UTC │
└────────┴──────┴───────┴─────┴───┴────────┴──────────┴───────┴──────────────────────

## Exercise 4: Advanced Queries

### 4.1 Find the most popular aircraft types (by number of flights)

Join flights with planes to get manufacturer and model information. Show top 10.

In [33]:
result = ctx.execute(
    """
    WITH joined AS (
        SELECT
            f.tailnum,
            p.type        AS aircraft_type,
            p.manufacturer,
            p.model
        FROM flights AS f
        INNER JOIN planes AS p
          ON p.tailnum = f.tailnum
        WHERE f.tailnum IS NOT NULL
          AND p.manufacturer IS NOT NULL
          AND p.model IS NOT NULL
    )
    SELECT
        aircraft_type,
        manufacturer,
        model,
        COUNT(*) AS flights_count
    FROM joined
    GROUP BY aircraft_type, manufacturer, model
    ORDER BY flights_count DESC, manufacturer, model
    LIMIT 10
    """
)

print(result)

shape: (10, 4)
┌─────────────────────────┬───────────────────────────────┬─────────────────┬───────────────┐
│ aircraft_type           ┆ manufacturer                  ┆ model           ┆ flights_count │
│ ---                     ┆ ---                           ┆ ---             ┆ ---           │
│ str                     ┆ str                           ┆ str             ┆ u32           │
╞═════════════════════════╪═══════════════════════════════╪═════════════════╪═══════════════╡
│ Fixed wing multi engine ┆ AIRBUS                        ┆ A320-232        ┆ 31278         │
│ Fixed wing multi engine ┆ EMBRAER                       ┆ EMB-145LR       ┆ 28027         │
│ Fixed wing multi engine ┆ EMBRAER                       ┆ ERJ 190-100 IGW ┆ 23716         │
│ Fixed wing multi engine ┆ AIRBUS INDUSTRIE              ┆ A320-232        ┆ 14553         │
│ Fixed wing multi engine ┆ EMBRAER                       ┆ EMB-145XR       ┆ 14051         │
│ Fixed wing multi engine ┆ BOEING           

### 4.2 Analyze route performance

Find the top 10 routes (origin-destination pairs) with:
- Total number of flights
- Average departure delay
- Percentage of flights delayed more than 30 minutes

Include airport names, not just codes.

In [35]:
# Total number of flights
result = ctx.execute(
    """
    SELECT
        origin,
        dest,
        COUNT(*) AS flights_count
    FROM flights
    GROUP BY origin, dest
    ORDER BY flights_count DESC, origin, dest
    LIMIT 10
    """
)
print(f"Total number of flight: {result}")

Total number of flight: shape: (10, 3)
┌────────┬──────┬───────────────┐
│ origin ┆ dest ┆ flights_count │
│ ---    ┆ ---  ┆ ---           │
│ str    ┆ str  ┆ u32           │
╞════════╪══════╪═══════════════╡
│ JFK    ┆ LAX  ┆ 11262         │
│ LGA    ┆ ATL  ┆ 10263         │
│ LGA    ┆ ORD  ┆ 8857          │
│ JFK    ┆ SFO  ┆ 8204          │
│ LGA    ┆ CLT  ┆ 6168          │
│ EWR    ┆ ORD  ┆ 6100          │
│ JFK    ┆ BOS  ┆ 5898          │
│ LGA    ┆ MIA  ┆ 5781          │
│ JFK    ┆ MCO  ┆ 5464          │
│ EWR    ┆ BOS  ┆ 5327          │
└────────┴──────┴───────────────┘


In [36]:
# Average departure delay
result = ctx.execute(
    """
    SELECT
        origin,
        dest,
        COUNT(*)                         AS flights_count,
        ROUND(AVG(dep_delay), 2)         AS avg_dep_delay
    FROM flights
    GROUP BY origin, dest
    ORDER BY flights_count DESC, origin, dest
    LIMIT 10
    """
)
print(f"Average departure delay: {result}")

Average departure delay: shape: (10, 4)
┌────────┬──────┬───────────────┬───────────────┐
│ origin ┆ dest ┆ flights_count ┆ avg_dep_delay │
│ ---    ┆ ---  ┆ ---           ┆ ---           │
│ str    ┆ str  ┆ u32           ┆ f64           │
╞════════╪══════╪═══════════════╪═══════════════╡
│ JFK    ┆ LAX  ┆ 11262         ┆ 8.52          │
│ LGA    ┆ ATL  ┆ 10263         ┆ 11.45         │
│ LGA    ┆ ORD  ┆ 8857          ┆ 10.74         │
│ JFK    ┆ SFO  ┆ 8204          ┆ 11.95         │
│ LGA    ┆ CLT  ┆ 6168          ┆ 8.97          │
│ EWR    ┆ ORD  ┆ 6100          ┆ 14.64         │
│ JFK    ┆ BOS  ┆ 5898          ┆ 11.69         │
│ LGA    ┆ MIA  ┆ 5781          ┆ 7.36          │
│ JFK    ┆ MCO  ┆ 5464          ┆ 10.6          │
│ EWR    ┆ BOS  ┆ 5327          ┆ 12.55         │
└────────┴──────┴───────────────┴───────────────┘


In [37]:
# Percentage of flights delayed more than 30 minutes
result = ctx.execute(
    """
    WITH route_stats AS (
        SELECT
            origin,
            dest,
            COUNT(*) AS flights_count,
            AVG(dep_delay) AS avg_dep_delay,
            SUM(CASE WHEN dep_delay > 30 THEN 1 ELSE 0 END) AS delayed_over_30
        FROM flights
        GROUP BY origin, dest
    )
    SELECT
        rs.origin,
        ao.name AS origin_name,
        rs.dest,
        ad.name AS dest_name,
        rs.flights_count,
        ROUND(rs.avg_dep_delay, 2) AS avg_dep_delay,
        ROUND( (rs.delayed_over_30 * 100.0) / rs.flights_count, 2) AS pct_delayed_gt_30
    FROM route_stats AS rs
    LEFT JOIN airports AS ao ON ao.faa = rs.origin
    LEFT JOIN airports AS ad ON ad.faa = rs.dest
    ORDER BY rs.flights_count DESC, rs.origin, rs.dest
    LIMIT 10
    """
)
print(f"Percentage of flights delayed more than 30 minutes: {result}")

Percentage of flights delayed more than 30 minutes: shape: (10, 7)
┌────────┬────────────────┬──────┬────────────────┬───────────────┬───────────────┬────────────────┐
│ origin ┆ origin_name    ┆ dest ┆ dest_name      ┆ flights_count ┆ avg_dep_delay ┆ pct_delayed_gt │
│ ---    ┆ ---            ┆ ---  ┆ ---            ┆ ---           ┆ ---           ┆ _30            │
│ str    ┆ str            ┆ str  ┆ str            ┆ u32           ┆ f64           ┆ ---            │
│        ┆                ┆      ┆                ┆               ┆               ┆ f64            │
╞════════╪════════════════╪══════╪════════════════╪═══════════════╪═══════════════╪════════════════╡
│ JFK    ┆ John F Kennedy ┆ LAX  ┆ Los Angeles    ┆ 11262         ┆ 8.52          ┆ 9.83           │
│        ┆ Intl           ┆      ┆ Intl           ┆               ┆               ┆                │
│ LGA    ┆ La Guardia     ┆ ATL  ┆ Hartsfield     ┆ 10263         ┆ 11.45         ┆ 12.25          │
│        ┆              

## Bonus: Compare with Polars

### Choose one of the queries above and implement it using Polars

This will help you understand the relationship between SQL and Polars operations.

In [40]:
# Example: Let's implement Exercise 2.1 (average delay by origin) in Polars

# SQL version (for reference)
sql_result = ctx.execute("""
    SELECT
        origin,
        AVG(dep_delay) as avg_delay
    FROM flights
    WHERE dep_delay IS NOT NULL
    GROUP BY origin
    ORDER BY avg_delay DESC
""")

# Polars version
polars_result = (
    flights
    .filter(pl.col('dep_delay').is_not_null())
    .group_by('origin')
    .agg(pl.col('dep_delay').mean().alias('avg_delay'))
    .sort('avg_delay', descending=True)
)

print("SQL Result:")
print(sql_result)
print("\nPolars Result:")
print(polars_result)

# Now implement one of your own queries in Polars below:
# Your Polars code here – Exercise 1.2

# SQL Version
result = ctx.execute("""
    SELECT dest, COUNT(*) as flight_count
    FROM flights
    GROUP BY dest
    ORDER BY flight_count DESC
    LIMIT 10
""")

print(result)

# Polars Version

top_destinations = (
    flights
    .group_by("dest")
    .agg(pl.len().alias("flight_count"))  # COUNT(*)
    .sort(["flight_count", "dest"], descending=[True, False])
    .head(10)
)

print(top_destinations)


SQL Result:
shape: (3, 2)
┌────────┬───────────┐
│ origin ┆ avg_delay │
│ ---    ┆ ---       │
│ str    ┆ f64       │
╞════════╪═══════════╡
│ EWR    ┆ 15.107954 │
│ JFK    ┆ 12.112159 │
│ LGA    ┆ 10.346876 │
└────────┴───────────┘

Polars Result:
shape: (3, 2)
┌────────┬───────────┐
│ origin ┆ avg_delay │
│ ---    ┆ ---       │
│ str    ┆ f64       │
╞════════╪═══════════╡
│ EWR    ┆ 15.107954 │
│ JFK    ┆ 12.112159 │
│ LGA    ┆ 10.346876 │
└────────┴───────────┘
shape: (10, 2)
┌──────┬──────────────┐
│ dest ┆ flight_count │
│ ---  ┆ ---          │
│ str  ┆ u32          │
╞══════╪══════════════╡
│ ORD  ┆ 17283        │
│ ATL  ┆ 17215        │
│ LAX  ┆ 16174        │
│ BOS  ┆ 15508        │
│ MCO  ┆ 14082        │
│ CLT  ┆ 14064        │
│ SFO  ┆ 13331        │
│ FLL  ┆ 12055        │
│ MIA  ┆ 11728        │
│ DCA  ┆ 9705         │
└──────┴──────────────┘
shape: (10, 2)
┌──────┬──────────────┐
│ dest ┆ flight_count │
│ ---  ┆ ---          │
│ str  ┆ u32          │
╞══════╪════════════